# Notebook 02 — Azure SQL Integration

**Project:** Card Transaction Fraud Monitoring  
**Platform:** Databricks + PySpark + Azure SQL Database

## Objective
Connect Databricks to Azure SQL, read the three master datasets, optionally load them into Azure SQL, and validate the existing SQL tables against the source CSV files.

### Datasets
| Dataset | Expected rows |
|---|---:|
| card_master | 124 |
| customer_master | 106 |
| merchant_master | 115 |

### Important execution setting
The SQL tables have already been loaded and verified for row counts. Therefore, `RUN_SQL_WRITE` is set to `False` by default. Keep it False for normal validation runs. Change it to True only when you intentionally want to overwrite the existing SQL table data.

## 1. Import libraries

In [0]:
from pyspark.sql import functions as F

## 2. Configure Azure SQL connection

Credentials are retrieved from the existing Databricks secret scope. Do not hard-code usernames or passwords.

In [0]:
SERVER = "capstone1dbserver.database.windows.net"
DATABASE = "capstone1team1creditcard"

JDBC_URL = (
    f"jdbc:sqlserver://{SERVER}:1433;"
    f"databaseName={DATABASE};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

CONNECTION_PROPERTIES = {
    "user": dbutils.secrets.get("azure-sql-scope", "sql-user"),
    "password": dbutils.secrets.get("azure-sql-scope", "sql-pwd"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("Azure SQL connection configuration completed.")

## 3. Configure source CSV paths

The master CSV files are located in the existing Unity Catalog volume under `bronze_schema`.

In [0]:
BASE_PATH = (
    "/Volumes/capstone_team01/bronze_schema/"
    "raw_data/datasets/master_data/azure_sql_seed"
)

TABLES = [
    "card_master",
    "customer_master",
    "merchant_master"
]

EXPECTED_COUNTS = {
    "card_master": 124,
    "customer_master": 106,
    "merchant_master": 115
}

# Keep False because the Azure SQL tables are already loaded.
# Set True only when intentionally replacing existing SQL data.
RUN_SQL_WRITE = False

## 4. Read master CSV datasets

In [0]:
csv_dfs = {}

for table in TABLES:
    file_path = f"{BASE_PATH}/{table}.csv"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    csv_dfs[table] = df

    actual_count = df.count()
    print(
        f"{table}: {actual_count} rows "
        f"(expected: {EXPECTED_COUNTS[table]})"
    )

## 5. Optional: Load master data into Azure SQL

This cell is disabled by default. It uses the SQL Server connector format that was used successfully in the existing integration notebook.

**Warning:** If enabled, this section overwrites the three target tables. Do not enable it just to perform validation.

In [0]:
if RUN_SQL_WRITE:
    for table in TABLES:
        print(f"Writing {table} to Azure SQL...")

        (
            csv_dfs[table].write
            .format("sqlserver")
            .mode("overwrite")
            .option("truncate", "true")
            .option("host", SERVER)
            .option("port", "1433")
            .option("database", DATABASE)
            .option("dbtable", f"dbo.{table}")
            .option("user", CONNECTION_PROPERTIES["user"])
            .option("password", CONNECTION_PROPERTIES["password"])
            .save()
        )

        print(f"Completed: dbo.{table}")
else:
    print("SQL write skipped. Existing Azure SQL tables are preserved.")

## 6. Read existing master tables from Azure SQL

In [0]:
sql_dfs = {}

for table in TABLES:
    df = spark.read.jdbc(
        url=JDBC_URL,
        table=f"dbo.{table}",
        properties=CONNECTION_PROPERTIES
    )

    sql_dfs[table] = df

    print(f"{table} SQL count: {df.count()}")

## 7. Compare CSV and Azure SQL row counts

In [0]:
print("========== ROW COUNT COMPARISON ==========")

count_results = {}

for table in TABLES:
    csv_count = csv_dfs[table].count()
    sql_count = sql_dfs[table].count()

    matched = csv_count == sql_count

    count_results[table] = matched

    print(
        f"{table} | CSV: {csv_count} | "
        f"SQL: {sql_count} | "
        f"{'MATCH' if matched else 'MISMATCH'}"
    )

## 8. Compare column names

In [0]:
print("========== COLUMN COMPARISON ==========")

column_results = {}

for table in TABLES:
    csv_columns = set(csv_dfs[table].columns)
    sql_columns = set(sql_dfs[table].columns)

    missing_in_sql = csv_columns - sql_columns
    extra_in_sql = sql_columns - csv_columns

    matched = not missing_in_sql and not extra_in_sql
    column_results[table] = matched

    print(f"\nTable: {table}")
    print("Missing in SQL:", missing_in_sql)
    print("Extra in SQL:", extra_in_sql)
    print("Column names:", "MATCH" if matched else "MISMATCH")

## 9. Normalize card master values for comparison

The earlier comparison showed that `card_master` had 124 records on each side, but all rows appeared different. The sample output showed values such as `2751.0` in CSV and `2751` in SQL.

This section compares `daily_limit` as a numeric value and date columns as dates. The remaining columns are compared as trimmed strings.

In [0]:
def normalize_card_data(df):
    return df.select(
        F.trim(F.col("card_id").cast("string")).alias("card_id"),
        F.trim(F.col("customer_id").cast("string")).alias("customer_id"),
        F.trim(F.col("card_type").cast("string")).alias("card_type"),
        F.trim(F.col("issuer_country").cast("string")).alias("issuer_country"),
        F.to_date(F.col("issue_date")).alias("issue_date"),
        F.to_date(F.col("expiry_date")).alias("expiry_date"),
        F.trim(F.col("card_status").cast("string")).alias("card_status"),
        F.trim(F.col("network").cast("string")).alias("network"),
        F.col("daily_limit").cast("double").alias("daily_limit"),
        F.trim(F.col("tokenized_flag").cast("string")).alias("tokenized_flag")
    )


def normalize_other_data(csv_df, sql_df):
    common_columns = [
        column for column in csv_df.columns
        if column in sql_df.columns
    ]

    csv_normalized = csv_df.select(*[
        F.trim(F.col(column).cast("string")).alias(column)
        for column in common_columns
    ])

    sql_normalized = sql_df.select(*[
        F.trim(F.col(column).cast("string")).alias(column)
        for column in common_columns
    ])

    return csv_normalized, sql_normalized

## 10. Compare actual records

`exceptAll` compares complete rows and preserves duplicate occurrences. A successful match requires zero records missing from SQL and zero extra records in SQL.

In [0]:
comparison_results = {}

print("========== RECORD COMPARISON ==========")

for table in TABLES:

    if table == "card_master":
        csv_compare = normalize_card_data(csv_dfs[table])
        sql_compare = normalize_card_data(sql_dfs[table])
    else:
        csv_compare, sql_compare = normalize_other_data(
            csv_dfs[table],
            sql_dfs[table]
        )

    missing_in_sql = csv_compare.exceptAll(sql_compare)
    extra_in_sql = sql_compare.exceptAll(csv_compare)

    missing_count = missing_in_sql.count()
    extra_count = extra_in_sql.count()

    matched = missing_count == 0 and extra_count == 0

    comparison_results[table] = {
        "missing_in_sql": missing_count,
        "extra_in_sql": extra_count,
        "matched": matched
    }

    print(f"\nTable: {table}")
    print("Missing records in SQL:", missing_count)
    print("Extra records in SQL:", extra_count)
    print("Data Match:", matched)

    if missing_count > 0:
        print("Sample records missing in SQL:")
        display(missing_in_sql.limit(5))

    if extra_count > 0:
        print("Sample extra records in SQL:")
        display(extra_in_sql.limit(5))

## 11. Final integration summary

In [0]:
print("========== FINAL SQL INTEGRATION SUMMARY ==========")

for table in TABLES:
    result = comparison_results[table]

    print(
        f"{table} | "
        f"CSV rows: {csv_dfs[table].count()} | "
        f"SQL rows: {sql_dfs[table].count()} | "
        f"Count match: {count_results[table]} | "
        f"Columns match: {column_results[table]} | "
        f"Missing: {result['missing_in_sql']} | "
        f"Extra: {result['extra_in_sql']} | "
        f"Data match: {result['matched']}"
    )

all_data_matched = all(
    comparison_results[table]["matched"]
    for table in TABLES
)

print("\nOverall record comparison:", "PASSED" if all_data_matched else "REVIEW REQUIRED")

## 12. Completion checklist

- [ ] Databricks connected to Azure SQL using the secret scope.
- [ ] All three master CSV files were read successfully.
- [ ] Existing Azure SQL tables were read successfully.
- [ ] CSV and SQL row counts were compared.
- [ ] Column names were compared.
- [ ] Actual records were compared after normalization.
- [ ] Any remaining mismatches were investigated.

**Next notebook:** `03_bronze_profiling`